# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedm0ssad/Machine-Learning-Intern/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)


## 1. My Lane: Refresh / Content Opportunity Scoring


- What it does: Ranks website pages to decide which ones editors should review first.
- Main goal: Identify pages that need updating, expanding, protecting, or removing.
- Why I chose it: In ML-01, my ML model achieved approximately 2.8× better Precision@50 than the manually written rule on a client-holdout test.
- Real-world value: It creates a ranked list of pages to review, with reasons explaining why each page was selected.
- Current limitation: The starter's is_declining_label uses the same 90-day period as the input features, making it an imperfect target.
- Future improvement: From Week 3 onward, warehouse daily history will help create a more reliable target using future data:
  - Input: Data from the previous 90 days.
  - Prediction: Whether the page will decline.
  - Target: Actual decline during the following 30 days.

In one sentence

I'm building an ML system that ranks website pages by their need for review and predicts future content decline so editors can prioritize their work.

In [ ]:
# Data anchor: the starter slice itself backs this lane - one row per page, enough volume to rank.
from pathlib import Path

candidates = [
    Path("../../data/raw/content_refresh_anonymized.csv"),  # run from work/notebooks (nbconvert default)
    Path("data/raw/content_refresh_anonymized.csv"),        # run from the repo root
]
csv_path = next(p for p in candidates if p.exists())

import pandas as pd

df = pd.read_csv(csv_path)
print(f"rows x cols: {df.shape[0]:,} x {df.shape[1]} | distinct clients: {df['client_id'].nunique()}")

rows x cols: 30,000 x 44 | distinct clients: 32


### 2. The Problem: Decision, Action, and Cost

- **Decision:** Decide which website pages editors should review first.
- **Who uses it:** Content editors who review the highest-priority pages each week.
- **Cost of a wrong decision:**
  - **False alarm:** An editor wastes time reviewing a page that doesn't need attention.
  - **Missed opportunity:** A page losing traffic continues to decline without intervention.
  - Missing an important, high-traffic page is more costly than missing a low-traffic page.
- **How to measure success:** Use **Precision@K** instead of accuracy.

**What is Precision@K?**

It measures how many of the top K recommended pages actually deserve review.

**Example:** If the model recommends 50 pages and 40 genuinely need review:

Precision@50 = 40 ÷ 50 = **80%**

### The Main Idea
**The goal isn't just to predict declining pages. It's to help editors spend their limited time reviewing the pages that matter most.**

In [ ]:
# Capacity math: the naive queue (all visible declining pages) is far beyond one editor's weekly capacity.
# impressions_90d >= 500 = the lane guide's "enough demand to matter" floor.
# trend_direction is the label side ONLY - it is never a feature (data-dictionary rule).

m_visible = df["impressions_90d"] >= 500
declining = df["trend_direction"] == "down"

n_visible = int(m_visible.sum())
n_visible_declining = int((m_visible & declining).sum())
weeks_of_queue = n_visible_declining / 50  # ~50 reviews per week

print(f"visible pages (impressions_90d >= 500): {n_visible:,}")
print(f"...of them declining: {n_visible_declining:,} ({declining[m_visible].mean() * 100:.1f}% of visible)")
print(f"at ~50 reviews/week, that queue alone is ~{weeks_of_queue:.0f} weeks -> ranking 'which first' is unavoidable")

visible pages (impressions_90d >= 500): 16,726
...of them declining: 9,961 (59.6% of visible)
at ~50 reviews/week, that queue alone is ~199 weeks -> ranking 'which first' is unavoidable


### 3. Understanding the Data: Why Is This Project Worth 7 Weeks?

**Three important numbers:**

1. **54.2% of pages are declining.**
   - More than half of the pages are losing performance.
   - Simply identifying declining pages isn't enough.
   - The real challenge is ranking them by priority.

2. **The top 10% of pages generate 80.5% of all clicks.**
   - A small number of pages attract most of the traffic.
   - Missing an important page could be costly.
   - That's why ranking quality matters more than overall accuracy.

3. **9,961 visible pages are already declining.**
   - These pages have at least 500 impressions over 90 days.
   - If an editor reviews 50 pages per week, reviewing all 9,961 would take approximately **200 weeks**.
   - Editors need the system to identify which pages deserve attention first.

### The Main Takeaway

**The problem isn't a lack of pages to improve. It's limited time to review them.**

Your ML model should rank pages so editors can focus on the most valuable opportunities first, using **Precision@K** to evaluate the recommendations.

In [ ]:
# The three numbers, computed live from the starter CSV.
n = len(df)

down_share = (df["trend_direction"] == "down").mean() * 100
top10 = df.sort_values("clicks_90d", ascending=False).head(n // 10)
click_share = top10["clicks_90d"].sum() / df["clicks_90d"].sum() * 100
m_visible = df["impressions_90d"] >= 500
queue = int((m_visible & (df["trend_direction"] == "down")).sum())

print(f"1) declining share of pages: {down_share:.1f}%")
print(f"2) share of all clicks_90d held by the top 10% of pages: {click_share:.1f}%")
print(f"3) visible pages (impressions_90d >= 500) that are declining: {queue:,}")

1) declining share of pages: 54.2%
2) share of all clicks_90d held by the top 10% of pages: 80.5%
3) visible pages (impressions_90d >= 500) that are declining: 9,961


### 4. What I Can and Cannot Claim

#### ✅ What I CAN claim

1. **Observed / Measured**
   - Report actual numbers and comparisons from the 30,000-page anonymized dataset.
   - Example: 54.2% of pages show declining impressions over the last 30 days.

2. **Directional Results**
   - In ML-01, the learned ranking achieved better Precision@20 and Precision@50 than the hand-written rule on a client-holdout test.
   - This is evidence from one experiment, **not a guarantee** that the model will always perform better.

3. **Decision Support**
   - The model ranks pages that a human editor should review first.
   - The editor makes the final decision.
   - The model doesn't guarantee that updating a page will improve its performance.

#### ❌ What I CANNOT claim

1. **No Causal Proof**
   - I cannot claim that refreshing a page caused its traffic to recover.
   - The available data doesn't establish that relationship.

2. **I'm Not Predicting Google's Algorithm**
   - I cannot claim to know or predict Google's ranking algorithm.
   - I'm analyzing observed website performance data.

3. **The Starter Label Isn't Ground Truth**
   - `is_declining_label` is derived from `trend_direction` and `trend_pct`, which use the same time window as the input features.
   - This creates a limitation in the starter dataset.
   - The label is a **proxy**, not definitive proof of future decline.
   - These label-related columns must never be used as model features because that would introduce information leakage.

### The Main Takeaway

**My model supports human decisions using observed data. It doesn't prove causation, predict Google's algorithm, or guarantee future outcomes.**

In [ ]:
# Label-side reality check (counts only - these columns are the label family, so they are never features).
# trend_pct is undefined when prev-30d impressions are 0 - the data dictionary itself says the label has holes.

print("trend_direction counts:", df["trend_direction"].value_counts().to_dict())
print("trend_pct rows blank (undefined when impressions_prev_30d = 0):", int(df["trend_pct"].isna().sum()))

trend_direction counts: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}
trend_pct rows blank (undefined when impressions_prev_30d = 0): 3388
